# 7.4. Multiple Input and Multiple Output Channels

So far, we treated images as 2-dimensional data. At first glance, this makes sense since each image has a given height and width. But pixels in colored images are composed of intensities across 3 base colors - red, green and blue. This gives colored images a 3rd dimension known as _channels_.

In this chapter, we'll see how convolutional layers handle such images. Fortunately, it's just a simple extension over how we handled 2-dimensional images - the key ideas remain the same with a few simple generalizations to our existing equations.

In [1]:
import mindspore

mindspore.set_device(device_target='Ascend', device_id=0)
mindspore.run_check()

/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/unify_schedule/vector/transdata/common/graph/transdata_graph_info.py:146: SyntaxWarning: invalid escape sequence '\c'
  2. In forward, tiling would not split c1 and c0, find c1\c0 based on t2.
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/unify_schedule/vector/transdata/common/graph/transdata_graph_info.py:172: SyntaxWarning: invalid escape sequence '\c'
  1. Forward: tiling would not split c1\c0\h0, find c1\c0\h1\h0 based on t2
/home/HwHiAiUser/.pyenv/versions/3.12.13/envs/orangepiaipro-20t/lib/python3.12/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/home/HwHiAiUser/.pyenv/versions/3.12.13/envs/orangep

MindSpore version:  2.8.0
The result of multiplication calculation is correct, MindSpore has been installed on platform [Ascend] successfully!


## 7.4.1. Multiple Input Channels

Given an RGB image with 3 channels, we can treat the image as 3 "sub-images" each with exactly 1 color - red, green or blue. Then, we compute the convolutions over each "sub-image" and sum up the results from each input channel. Note that this requires us to have 1 convolution kernel per channel, for a total of 3 kernels.

In the single input channel case, our convolution kernel had dimensions $k_h \times k_w$. With 3 input channels, we have 3 such kernels with the same dimensions $k_h \times k_w$. Mathematically, it's much more straightforward to treat the 3 kernels as a single tensor with dimensions $3 \times k_h \times k_w$. In general, when we have $c_i$ input channels, our combined convolution kernel has dimensions $c_i \times k_h \times k_w$.

Recall our convolution operation `corr2d` in chapter 7.2 implemented from first principles. Our original function only deals with a single input channel.

In [2]:
import mindspore.ops as ops

def corr2d(X, K):
    """Compute 2D cross-correlation."""
    h, w = K.shape
    Y = ops.zeros((X.shape[0] - h + 1, X.shape[1] - w + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j] = ops.sum(X[i:i+h, j:j+w] * K)
    return Y

Now we can define our convolution operation for images consisting of multiple channels. We simply compute the 2D convolution over each channel and sum the results together as implemented below.

In [3]:
def corr2d_multi_in(X, K):
    return sum(corr2d(x, k) for x, k in zip(X, K))

Let's see it in action. Given an image $\mathrm{X}$ with 2 channels.

$$
\begin{align}
\mathrm{X}_1 &= \begin{pmatrix} 0 & 1 & 2 \\ 3 & 4 & 5 \\ 6 & 7 & 8 \end{pmatrix} \\
\mathrm{X}_2 &= \begin{pmatrix} 1 & 2 & 3 \\ 4 & 5 & 6 \\ 7 & 8 & 9 \end{pmatrix}
\end{align}
$$

And a convolution kernel $\mathrm{K}$ with dimensions $2 \times 2 \times 2$, each sub-kernel as defined below.

$$
\begin{align}
\mathrm{K}_1 = \begin{pmatrix} 0 & 1 \\ 2 & 3 \end{pmatrix} \\
\mathrm{K}_2 = \begin{pmatrix} 1 & 2 \\ 3 & 4 \end{pmatrix}
\end{align}
$$

The convolution $\mathrm{X} \ast \mathrm{K} = \bigl( \begin{smallmatrix} 56 & 72 \\ 104 & 120 \end{smallmatrix} \bigr)$. For example, the resulting pixel to the top left of the feature map is given by $(0 \times 0 + 1 \times 1 + 3 \times 2 + 4 \times 3) + (1 \times 1 + 2 \times 2 + 4 \times 3 + 5 \times 4) = 19 + 37 = 56$.

In [4]:
X_1 = ops.reshape(ops.arange(9), (3, 3))
X_2 = X_1 + 1
K_1 = ops.reshape(ops.arange(4), (2, 2))
K_2 = K_1 + 1
X = ops.stack([X_1, X_2])
K = ops.stack([K_1, K_2])
X, K, X.shape, K.shape

/usr/local/Ascend/cann-8.5.0/python/site-packages/asc_op_compile_base/asc_op_compiler/ascendc_compile_gen_code.py:161: SyntaxWarning: invalid escape sequence '\w'
  match = re.search(f'{option}=(\w+)', ' '.join(compile_options))
2026-05-02 23:14:27.765084: E external/org_tensorflow/tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute is_closed which is not in the op definition: Op<name=Range; signature=start:Tidx, limit:Tidx, delta:Tidx -> output:Tidx; attr=Tidx:type,default=DT_INT32,allowed=[DT_BFLOAT16, DT_HALF, DT_FLOAT, DT_DOUBLE, DT_INT8, DT_INT16, DT_INT32, DT_INT64, DT_UINT16, DT_UINT32]> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node Range1}}
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/d

(Tensor(shape=[2, 3, 3], dtype=Int64, value=
 [[[0, 1, 2],
   [3, 4, 5],
   [6, 7, 8]],
  [[1, 2, 3],
   [4, 5, 6],
   [7, 8, 9]]]),
 Tensor(shape=[2, 2, 2], dtype=Int64, value=
 [[[0, 1],
   [2, 3]],
  [[1, 2],
   [3, 4]]]),
 (2, 3, 3),
 (2, 2, 2))

In [5]:
from mindspore import dtype as mstype

Y = corr2d_multi_in(X, K).astype(dtype=mstype.int64)
Y, Y.shape

(Tensor(shape=[2, 2], dtype=Int64, value=
 [[ 56,  72],
  [104, 120]]),
 (2, 2))

## 7.4.2. Multiple Output Channels

Notice in the previous section that our feature map was a simple $2 \times 2$ matrix, despite our input image and convolution kernel being 3-dimensional tensors.

Just as we introduced multiple _input_ channels with colored images, we can also have multiple _output_ channels for our resulting feature map, giving our feature map dimensions of the form $c_o \times h \times w$ where $c_o$ is the number of output channels.

To accommodate for multiple output channels, we add yet another dimension to our convolution kernel, resulting in a 4-dimensional tensor with dimensions $c_o \times c_i \times k_h \times k_w$. The resulting feature map is obtained by stacking the results of each of $c_o$ cross-correlation operations between our input image $\mathrm{X}_{c_i \times h \times w}$ and convolution sub-kernel $\mathrm{K}_{c_i \times k_h \times k_w}$. Each cross-correlation operation has dimensions $(h - k_h + 1) \times (w - k_w + 1)$ and we have $c_o$ of them so the resulting feature map has dimensions $c_o \times (h - k_h + 1) \times (w - k_w + 1)$.

In [6]:
def corr2d_multi_in_out(X, K):
    return ops.stack([corr2d_multi_in(X, k) for k in K])

Let's run a simple example to verify our intuition. We won't care about the specific values but let's check that the input and output shapes are correct.

Suppose we have an image $\mathrm{X}$ of size $10 \times 10$ pixels. Each pixel has 3 channels corresponding to RGB intensities. Our image then has dimensions $3 \times 10 \times 10$ - we have $c_i = 3$, $h = 10$ and $w = 10$.

Now suppose we pass our image $\mathrm{X}$ through a convolutional layer with kernel $\mathrm{K}$ of size $6 \times 3 \times 5 \times 5$. This means we have $c_o = 6$, $k_h = 5$ and $k_w = 5$. Our resulting feature map $\mathrm{Y}$ should then have shape $6 \times (10 - 5 + 1) \times (10 - 5 + 1) = 6 \times 6 \times 6$.

Let's run our examples through `corr2d_multi_in_out` to see if that is the case.

In [7]:
X = ops.randn(3, 10, 10)
K = ops.randn(6, 3, 5, 5)
Y = corr2d_multi_in_out(X, K)
Y.shape == (6, 6, 6)

True

The expression `Y.shape == (6, 6, 6)` evaluates to `True`. Great!

## 7.4.3. $1 \times 1$ Convolutional Layer

For simple cases in previous chapters where we only had 1 input channel and 1 output channel, a $1 \times 1$ convolutional layer does not make sense since such a kernel is unable to correlate neighboring pixels. However, with $c_i > 1$, such a convolution kernel can be used to operate on the input channel dimension $c_i$ to learn linear relationships between the input channels for each pixel. With $c_o > 1$, we simply repeat this process $c_o$ times and stack the results on the 1st dimension of our resulting feature map.

When we have a kernel $\mathrm{K}$ with dimensions $c_o \times c_i \times 1 \times 1$, the cross-correlation operation $\mathrm{X} \ast \mathrm{K}$ simply degenerates to a fully connected layer operating on each individual pixel with $c_i$ input channels and $c_o$ output channels. Intuitively, this makes sense since for each input channel, the cross-correlation operation of the input pixel $x_{i} = \mathrm{X}_{ijk}$ with a $1 \times 1$ sub-kernel $k_{i} = \mathrm{K}_{ijk}$ is simply the product of two scalars $x_{i} k_{i}$. Then, we have:

$$
\begin{align}
\mathbf{x} &= \begin{pmatrix} x_1 \\ \dots \\ x_{c_i} \end{pmatrix} \\
\mathbf{k} &= \begin{pmatrix} k_1 \\ \dots \\ k_{c_i} \end{pmatrix} \\
y &= \sum_{l=1}^{c_i} x_l k_l \\
&= \langle \mathbf{x}, \mathbf{k} \rangle \\
&= \mathbf{x}^{\mathrm{T}} \mathbf{k}
\end{align}
$$

This is basically the product of two matrices without added bias, exactly how fully connected layers work.

Let's verify our intution by defining $1 \times 1$ convolutions with matrix multiplication in our function `corr2d_multi_in_out_1x1` and compare our results with our general implementation `corr2d_multi_in_out` for the $1 \times 1$ case. Since both implementations are equivalent, we expect the difference between both results to be effectively zero, absent a negligible error term $\epsilon \approx 1 \times 10^{-3}$ due to precision issues with half-precision floating point calculations \(FP16\).

In [8]:
def corr2d_multi_in_out_1x1(X, K):
    c_i, h, w = X.shape
    c_o = K.shape[0]
    X = ops.reshape(X, (c_i, h * w))
    K = ops.reshape(K, (c_o, c_i))
    Y = ops.matmul(K, X)
    return ops.reshape(Y, (c_o, h, w))

X = ops.randn(3, 3, 3, dtype=mstype.float16)
K = ops.randn(2, 3, 1, 1, dtype=mstype.float16)
Y_0 = corr2d_multi_in_out_1x1(X, K)
Y_1 = corr2d_multi_in_out(X, K)
Y_0.shape, Y_1.shape, Y_0.shape == Y_1.shape

((2, 3, 3), (2, 3, 3), True)

In [9]:
import mindspore.nn as nn

loss_fn = nn.MSELoss(reduction='sum')
loss = loss_fn(Y_0, Y_1).item()
print(f'Loss: {loss:.4f}')
loss < 1e-3

Loss: 0.0000


True

## 7.4.4. Discussion

We saw how convolutions can be applied to colored images with multiple input channels and produce feature maps with multiple output channels. We also saw how $1 \times 1$ convolutions are effectively fully connected layers applied across the input channel dimension and computed per pixel. This gives us the best of both worlds - we can exploit _locality_ and _translation invariance_ of image features with convolutions, all the while learning linear relationships between input channels per pixel with $1 \times 1$ kernels.

In the next chapter, we'll see how _pooling_ allows us to learn non-linear relationships among input channels after passing through convolutional layers, similar to how activation functions such as ReLU allows us to learn hidden representations of our data by introducing non-linearities after passing through fully connected layers.